# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasleen13/ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/jasleen13/ML-Internship.git"
DATA_REL = Path("data") / "raw" / "content_refresh_anonymized.csv"

def find_repo_root(start: Path):
    p = start
    while not (p / DATA_REL).exists() and p != p.parent:
        p = p.parent
    return p if (p / DATA_REL).exists() else None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    clone_dir = Path("/content/ML-Internship") if Path("/content").exists() else Path.cwd() / "ML-Internship"
    if not (clone_dir / DATA_REL).exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

DATA_PATH = repo_root / DATA_REL
OUTPUTS_DIR = repo_root / "work" / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Using data at:", DATA_PATH)
print(df.shape)

Using data at: /content/ML-Internship/data/raw/content_refresh_anonymized.csv
(30000, 44)


**The rule, in plain words:** a page is worth reviewing for its title/meta/snippet if it gets
enough search visibility to matter, ranks well enough that clicks should be flowing (top 20),
and its CTR sits below what other pages at the same position tier are earning. The gap between
observed and expected CTR, scaled by how many impressions are riding on it, is the score, bigger
gap and bigger audience means review it first.

**Reason code:** `low_ctr_visible_page`, borrowed directly from the flag the FlyRank session
walked through (`impressions_90d >= 500`, position 1 to 20, CTR below the tier's own median).

**Action label:** `rewrite_title_meta`.

Before coding that up, two signals the rule leans on, checked first.

### Signal 1: staleness, behind the refresh flags (`stale_visible_page`)
Does `days_since_last_update >= 180` actually associate with worse performance? Two bucket
tables: the full dataset, then the same buckets restricted to visible pages
(`impressions_90d >= 500`), since that's the volume floor the actual flag uses.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Freshness bucket table, ALL rows (n per bucket):")
all_rows = df.groupby("freshness_tier").agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"),
                                             mean_impr=("impressions_90d", "mean")).sort_index()
print(all_rows)
print()

print("Same buckets, visible pages only (impressions_90d >= 500):")
visible_only = df[df["impressions_90d"] >= 500]
vis_rows = visible_only.groupby("freshness_tier").agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"),
                                                        mean_impr=("impressions_90d", "mean")).sort_index()
print(vis_rows)

Freshness bucket table, ALL rows (n per bucket):
                    n  mean_ctr    mean_impr
freshness_tier                              
0-30            20480  0.609021  4199.614062
181+              174  3.693276  1172.448276
31-90             175  0.117543  6506.748571
91-180           9171  0.238367  7486.665140

Same buckets, visible pages only (impressions_90d >= 500):
                    n  mean_ctr     mean_impr
freshness_tier                               
0-30            10063  0.271484   8436.230051
181+               17  0.208824  11600.647059
31-90              88  0.144886  12711.272727
91-180           6558  0.249686  10399.451052


**Verdict: MIXED.** In the unfiltered table, `181+` shows a much *higher* mean CTR than fresh
content (3.69 vs 0.61), the opposite of what the flag assumes, but that bucket has n=174 and
the number is almost certainly a volume artifact, the same trap Discovery B caught back in
Week 1: a handful of low-impression pages where one click swings the percentage wildly. Once
the volume floor is applied (the same floor the real flag uses), the direction flips to match
the hypothesis, stale visible pages average 0.21% CTR against 0.27% for fresh visible pages, but
now n=17, too thin to call this CONFIRMED. Staleness is directionally plausible but not proven
at a sample size worth trusting, so it stays out of the score below. That's a legitimate result,
not a failure to find something.

### Signal 2: CTR-vs-position, behind the CTR-fix logic (`low_ctr_visible_page`)
Does CTR fall as position gets worse? Same volume-floored bucket table from Discovery B, rerun
here as this week's verification.

In [4]:
floor = df[df["impressions_90d"] >= 100]
tier_table = floor.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print("CTR by position tier (impressions_90d >= 100):")
print(tier_table)


CTR by position tier (impressions_90d >= 100):
                   mean  count
position_tier                 
page_1         0.354760   8633
top_3          0.334128    533
striking       0.255782   5903
page_3_5       0.142359   6058
deep           0.055415    879


**Verdict: CONFIRMED.** CTR steps down cleanly and monotonically from 0.35% at `page_1` to
0.055% at `deep`, every tier carries a real sample (n=879 to n=8,633). This is the signal the
rule below is built on.


## 2. Build the ranked queue (writes the CSV)

Candidate pool: visible (impressions_90d >= 500) and ranking well enough that a click should be realistic (0 < avg_position <= 20), matching the flag's own volume and position bounds. Score: the CTR gap below the tier median, scaled by log1p(impressions_90d) so a big audience outweighs a small one without letting the largest pages swamp everything linearly. Only observed, same-window signals go in, trend_direction and trend_pct are never touched, per the label trap documented in the data dictionary.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible = df["impressions_90d"] >= 500
in_range = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
lane = df[visible & in_range].copy()

tier_median_ctr = lane.groupby("position_tier")["ctr"].transform("median")
lane["ctr_gap"] = lane["ctr"] - tier_median_ctr
lane["score"] = (lane["ctr_gap"] < 0).astype(int) * (-lane["ctr_gap"]).clip(lower=0) * np.log1p(lane["impressions_90d"])
lane["reason_code"] = "low_ctr_visible_page"
lane["action"] = "rewrite_title_meta"

queue = lane.sort_values("score", ascending=False).reset_index(drop=True)
csv_path = OUTPUTS_DIR / "baseline_action_score.csv"
queue[["content_id", "client_id", "position_tier", "impressions_90d", "avg_position",
       "ctr", "ctr_gap", "score", "reason_code", "action"]].to_csv(csv_path, index=False)

print(f"Candidate pool: {len(lane):,} of {len(df):,} pages")
print(f"Wrote ranked queue to: {csv_path}")
print()
print(queue["score"].describe())

Candidate pool: 12,023 of 30,000 pages
Wrote ranked queue to: /content/ML-Internship/work/outputs/baseline_action_score.csv

count    12023.000000
mean         0.454235
std          0.590355
min          0.000000
25%          0.000000
50%          0.000000
75%          0.903423
max          2.939653
Name: score, dtype: float64


In [6]:
import json

metrics = {
    "lane": "Lane 4: CTR / Engagement Opportunity Scoring",
    "candidate_pool_size": int(len(lane)),
    "total_pages": int(len(df)),
    "reason_code": "low_ctr_visible_page",
    "action_label": "rewrite_title_meta",
    "signal_1_staleness": {
        "verdict": "MIXED",
        "note": "unfiltered direction is opposite (n=174, volume artifact); volume-floored direction matches the flag's assumption but n=17 is too thin to confirm; excluded from the score."
    },
    "signal_2_ctr_vs_position": {
        "verdict": "CONFIRMED",
        "note": "monotonic CTR decline by position tier, n=879 to n=8,633, this drives the score."
    },
    "score_stats": {
        "min": float(lane["score"].min()),
        "median": float(lane["score"].median()),
        "max": float(lane["score"].max()),
        "n_scored_above_zero": int((lane["score"] > 0).sum())
    }
}

metrics_path = OUTPUTS_DIR / "w04_baseline_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Wrote metrics receipt to: {metrics_path}")
print(json.dumps(metrics, indent=2))

Wrote metrics receipt to: /content/ML-Internship/work/outputs/w04_baseline_metrics.json
{
  "lane": "Lane 4: CTR / Engagement Opportunity Scoring",
  "candidate_pool_size": 12023,
  "total_pages": 30000,
  "reason_code": "low_ctr_visible_page",
  "action_label": "rewrite_title_meta",
  "signal_1_staleness": {
    "verdict": "MIXED",
    "note": "unfiltered direction is opposite (n=174, volume artifact); volume-floored direction matches the flag's assumption but n=17 is too thin to confirm; excluded from the score."
  },
  "signal_2_ctr_vs_position": {
    "verdict": "CONFIRMED",
    "note": "monotonic CTR decline by position tier, n=879 to n=8,633, this drives the score."
  },
  "score_stats": {
    "min": 0.0,
    "median": 0.0,
    "max": 2.939652591623707,
    "n_scored_above_zero": 5888
  }
}


## 3. Top-10 review

Every one of the top 10 is a `keyword article`, ranking in the 4 to 10 range (well inside
page 1), with impressions in the tens or hundreds of thousands and CTR at 0.00% to 0.03%, near
total non-clicks despite strong visibility. One line each: the action, why it's there, and what
would make it wrong.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)
print(top10[["content_id", "content_type", "position_tier", "impressions_90d", "clicks_90d",
              "avg_position", "ctr", "ctr_gap", "score"]].to_string(index=False))


          content_id    content_type position_tier  impressions_90d  clicks_90d  avg_position  ctr  ctr_gap    score
content_c8e9d6ab9013 keyword article        page_1           208678           0           9.7 0.00    -0.24 2.939653
content_453722754fea keyword article        page_1           140079          16           7.6 0.01    -0.23 2.725493
content_39881853ef0c keyword article        page_1           112434          10           7.2 0.01    -0.23 2.674930
content_c84a0ab98e90 keyword article        page_1           223271          70           7.8 0.03    -0.21 2.586391
content_0919dd345d80 keyword article        page_1           119217          26           7.0 0.02    -0.22 2.571516
content_d274ac4158ef keyword article        page_1            65138           6           6.8 0.01    -0.23 2.549384
content_e5f459e737b7 keyword article        page_1            56363           3           5.9 0.01    -0.23 2.516105
content_c1fe78bc4e37 keyword article        page_1           134

1. **`content_c8e9d6ab9013`**, action `rewrite_title_meta`: 208,678 impressions at position
   9.7 with **0 clicks**. Why it's here: the single largest impression count in the queue paired
   with a literal zero, the biggest possible gap. What would make it wrong: this is thin enough
   to be a tracking or attribution bug rather than a real title problem, worth a manual GSC check
   before touching the page at all.
2. **`content_453722754fea`**: 140,079 impressions, 16 clicks, position 7.6. Why it's here: high
   volume, CTR far below the tier median. What would make it wrong: 2,700 words suggests a
   substantial page, if it already has a strong meta description, the real issue may be intent
   mismatch rather than snippet wording, worth a manual read before rewriting.
3. **`content_39881853ef0c`**: 112,434 impressions, 10 clicks, position 7.2. Why it's here: same
   client as #2 (`client_f369cb89fc`), a second data point suggesting a client-wide pattern, not
   a one-off page. What would make it wrong: if the whole client's titles follow one template,
   the fix might be one style change applied broadly, not ten separate rewrites.
4. **`content_c84a0ab98e90`**: 223,271 impressions, 70 clicks, position 7.8. Why it's here:
   highest impression count with a nonzero click count, so at least clickable, ctr still far
   under tier median. What would make it wrong: 70 clicks off 223k impressions could reflect a
   rich SERP result (image pack, video carousel) siphoning attention, not a bad title.
5. **`content_0919dd345d80`**: 119,217 impressions, 26 clicks, position 7.0. Why it's here:
   strong position (top of page 1) makes the low CTR more surprising, not less. What would make
   it wrong: same client as #6 below, worth checking whether this is a client-level pattern too.
6. **`content_d274ac4158ef`**: 65,138 impressions, 6 clicks, position 6.8, shortest article in
   the top 10 (1,428 words). Why it's here: good position, thin CTR. What would make it wrong:
   the short length itself could be the real issue (thin content, not a bad snippet), a content
   expansion might matter more than a title rewrite.
7. **`content_e5f459e737b7`**: 56,363 impressions, 3 clicks, position 5.9, `transactional`
   intent. Why it's here: transactional intent pages usually earn higher CTR than informational
   ones at the same position, this one is badly underperforming that expectation. What would
   make it wrong: if the title already promises the transaction clearly, the drop-off might be
   snippet-level (missing price, rating stars, or review count) rather than the title itself.
8. **`content_c1fe78bc4e37`**: 134,055 impressions, 43 clicks, position 7.5, `commercial` intent,
   `word_count` missing. Why it's here: high volume, low CTR, and it's one of only two rows in
   the top 10 with a blank word count. What would make it wrong: missing word_count is a data
   gap, not a content problem, worth confirming the page actually exists and is indexed before
   assuming the title is at fault.
9. **`content_339b357d04c7`**: 46,879 impressions, 7 clicks, position 3.7, the best position in
   the entire top 10. Why it's here: this close to `top_3`, CTR this low is the most surprising
   entry in the list. What would make it wrong: pages this close to top_3 sometimes lose clicks
   to a featured snippet or "People Also Ask" box stealing the top of the results page, a title
   rewrite wouldn't fix that.
10. **`content_65114d89496d`**: 72,631 impressions, 12 clicks, position 6.5, `transactional`
    intent, `word_count` missing. Why it's here: same pattern as #7 and #8, transactional intent
    underperforming its tier, plus a missing word count worth double-checking before acting.

## 4. Weak picks + leakage check

**Weakest pick: `content_c8e9d6ab9013` (#1).** A literal zero out of 208,678 impressions is an
extreme value, and extreme values are exactly where a data or tracking artifact hides rather
than a real content problem, per the same lesson Discovery B taught with the `feedly article`
outlier back in Week 1. It's the single largest score in the queue precisely because it's the
single most extreme row, that combination is worth a manual sanity check (does this URL even
resolve? is it indexed?) before it goes to an editor's desk.

**Second weakest: `content_339b357d04c7` (#9).** Position 3.7 is close enough to `top_3` that
SERP features (snippets, People Also Ask, image packs) are a live alternative explanation the
score can't see, since none of that is in this dataset. A title rewrite is the wrong action if
the real problem is a SERP feature above the organic result.

**Leakage check:** confirm the score touches no future window and no label-derived column.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
LEAKY_COLUMNS = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
                  "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}
SCORE_INPUT_COLUMNS = {"impressions_90d", "avg_position", "ctr", "position_tier"}

overlap = LEAKY_COLUMNS & SCORE_INPUT_COLUMNS
print(f"Leaky columns touched by the score: {overlap if overlap else 'none'}")
print(f"Score built only from: {sorted(SCORE_INPUT_COLUMNS)}")
print("Reason code and action label are hand-written strings, not derived from any label column.")

Leaky columns touched by the score: none
Score built only from: ['avg_position', 'ctr', 'impressions_90d', 'position_tier']
Reason code and action label are hand-written strings, not derived from any label column.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.